# 初步数据探索 — Taobao UserBehavior

**目标**:看清楚原始 CSV 长什么样,为后续分析(切口 A/B/D)做准备。

**本 notebook 只做这些**:
1. 文件元信息(行数、字段、类型)
2. 抽样预览
3. 每列唯一值数
4. 缺失值检查
5. 行为类型分布
6. 时间范围与每日记录数

**不做**:任何留存、漏斗、用户分群、画图。那是下一步的事。

---

**数据集来源**:[阿里天池 Taobao UserBehavior](https://tianchi.aliyun.com/dataset/649)

**字段定义**(根据天池文档):

| 字段 | 类型 | 含义 |
|---|---|---|
| user_id | int | 用户 ID(脱敏后) |
| item_id | int | 商品 ID |
| category_id | int | 商品类目 ID |
| behavior_type | str | 行为:pv(浏览) / cart(加购) / fav(收藏) / buy(购买) |
| timestamp | int | Unix epoch 秒级时间戳 |

CSV **无表头**,column 顺序按上表。

## 0. 环境与文件准备

In [ ]:
import duckdb
from pathlib import Path

DATA_PATH = Path('../data/raw/ODS/UserBehavior.csv').resolve()
print(f'数据文件: {DATA_PATH}')
print(f'文件存在: {DATA_PATH.exists()}')
print(f'文件大小: {DATA_PATH.stat().st_size / (1024**3):.2f} GB')

con = duckdb.connect()  # 内存数据库,不持久化
print(f'DuckDB 版本: {duckdb.__version__}')

## 1. 文件第一眼:确认 CSV 格式

天池版本的 CSV 无表头,先读前 5 行原始文本再次确认,免得 schema 对错列。

In [2]:
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for i in range(5):
        print(f'行 {i+1}: {f.readline().strip()}')

行 1: 1,2268318,2520377,pv,1511544070
行 2: 1,2333346,2520771,pv,1511561733
行 3: 1,2576651,149192,pv,1511572885
行 4: 1,3830808,4181361,pv,1511593493
行 5: 1,4365585,2520377,pv,1511596146


**期望**:每行是 `user_id,item_id,category_id,behavior_type,timestamp` 五个值,逗号分隔。
如果第一行竟然是字段名,把下面 cell 里的 `header=False` 改成 `header=True`。

## 2. 建立 DuckDB 视图

把 CSV 注册为视图 `user_behavior`,后面所有查询都通过它。
显式指定列名和类型,不依赖 DuckDB 自动推断(因为无表头,自动推断会得到 column0/column1...)。

In [3]:
con.execute(f"""
CREATE OR REPLACE VIEW user_behavior AS
SELECT * FROM read_csv(
    '{DATA_PATH.as_posix()}',
    header=False,
    columns={{
        'user_id': 'BIGINT',
        'item_id': 'BIGINT',
        'category_id': 'BIGINT',
        'behavior_type': 'VARCHAR',
        'timestamp': 'BIGINT'
    }}
)
""")
print('视图 user_behavior 创建完成')

视图 user_behavior 创建完成


## 3. 字段结构

In [4]:
con.sql("DESCRIBE user_behavior").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ user_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ item_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ behavior_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ timestamp     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



## 4. 抽样预览(前 10 行)

In [5]:
con.sql("SELECT * FROM user_behavior LIMIT 10").show()

┌─────────┬─────────┬─────────────┬───────────────┬────────────┐
│ user_id │ item_id │ category_id │ behavior_type │ timestamp  │
│  int64  │  int64  │    int64    │    varchar    │   int64    │
├─────────┼─────────┼─────────────┼───────────────┼────────────┤
│       1 │ 2268318 │     2520377 │ pv            │ 1511544070 │
│       1 │ 2333346 │     2520771 │ pv            │ 1511561733 │
│       1 │ 2576651 │      149192 │ pv            │ 1511572885 │
│       1 │ 3830808 │     4181361 │ pv            │ 1511593493 │
│       1 │ 4365585 │     2520377 │ pv            │ 1511596146 │
│       1 │ 4606018 │     2735466 │ pv            │ 1511616481 │
│       1 │  230380 │      411153 │ pv            │ 1511644942 │
│       1 │ 3827899 │     2920476 │ pv            │ 1511713473 │
│       1 │ 3745169 │     2891509 │ pv            │ 1511725471 │
│       1 │ 1531036 │     2920476 │ pv            │ 1511733732 │
└─────────┴─────────┴─────────────┴───────────────┴────────────┘
  10 rows                

## 5. 总行数与每列唯一值

> 这一步会**全表扫描一次**,3.5GB 文件大约要 30-60 秒。

In [6]:
con.sql("""
SELECT
    COUNT(*)                       AS total_rows,
    COUNT(DISTINCT user_id)        AS unique_users,
    COUNT(DISTINCT item_id)        AS unique_items,
    COUNT(DISTINCT category_id)    AS unique_categories,
    COUNT(DISTINCT behavior_type)  AS distinct_behaviors
FROM user_behavior
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────┬──────────────┬───────────────────┬────────────────────┐
│ total_rows │ unique_users │ unique_items │ unique_categories │ distinct_behaviors │
│   int64    │    int64     │    int64     │       int64       │       int64        │
├────────────┼──────────────┼──────────────┼───────────────────┼────────────────────┤
│  100150807 │       987994 │      4162024 │              9439 │                  4 │
└────────────┴──────────────┴──────────────┴───────────────────┴────────────────────┘



**天池官方参考值**:
- 行数:约 1.0015 亿
- 用户数:约 98.8 万
- 商品数:约 416 万
- 类目数:约 9,440
- 行为类型:4 种(pv / cart / fav / buy)

实际数对得上即可。

## 6. 缺失值检查

对每列分别统计 NULL,字符串类型额外检查空字符串。

In [7]:
con.sql("""
SELECT
    SUM(CASE WHEN user_id       IS NULL THEN 1 ELSE 0 END) AS null_user_id,
    SUM(CASE WHEN item_id       IS NULL THEN 1 ELSE 0 END) AS null_item_id,
    SUM(CASE WHEN category_id   IS NULL THEN 1 ELSE 0 END) AS null_category_id,
    SUM(CASE WHEN behavior_type IS NULL OR behavior_type = '' THEN 1 ELSE 0 END) AS null_or_empty_behavior_type,
    SUM(CASE WHEN timestamp     IS NULL THEN 1 ELSE 0 END) AS null_timestamp
FROM user_behavior
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬──────────────┬──────────────────┬─────────────────────────────┬────────────────┐
│ null_user_id │ null_item_id │ null_category_id │ null_or_empty_behavior_type │ null_timestamp │
│    int128    │    int128    │      int128      │           int128            │     int128     │
├──────────────┼──────────────┼──────────────────┼─────────────────────────────┼────────────────┤
│            0 │            0 │                0 │                           0 │              0 │
└──────────────┴──────────────┴──────────────────┴─────────────────────────────┴────────────────┘



**期望**:全部为 0(天池数据集已清洗过)。
如果某列有缺失,后续分析需要决定是丢弃还是填补。

## 7. 行为类型分布

In [8]:
con.sql("""
SELECT
    behavior_type,
    COUNT(*)                                            AS rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS pct
FROM user_behavior
GROUP BY behavior_type
ORDER BY rows DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬──────────┬────────┐
│ behavior_type │   rows   │  pct   │
│    varchar    │  int64   │ double │
├───────────────┼──────────┼────────┤
│ pv            │ 89716264 │  89.58 │
│ cart          │  5530446 │   5.52 │
│ fav           │  2888258 │   2.88 │
│ buy           │  2015839 │   2.01 │
└───────────────┴──────────┴────────┘



**期望**:`pv` 占绝大多数(>80%),其余依次 `cart`、`fav`、`buy`。
如果出现意料外的 behavior_type 值,说明数据可能有脏行。

## 8. 时间范围

In [9]:
con.sql("""
SELECT
    MIN(timestamp)                              AS min_ts,
    MAX(timestamp)                              AS max_ts,
    to_timestamp(MIN(timestamp))                AS min_datetime,
    to_timestamp(MAX(timestamp))                AS max_datetime,
    (MAX(timestamp) - MIN(timestamp)) / 86400.0 AS days_span
FROM user_behavior
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬──────────────────────────┬──────────────────────────┬───────────────────┐
│   min_ts    │   max_ts   │       min_datetime       │       max_datetime       │     days_span     │
│    int64    │   int64    │ timestamp with time zone │ timestamp with time zone │      double       │
├─────────────┼────────────┼──────────────────────────┼──────────────────────────┼───────────────────┤
│ -2134949234 │ 2122867355 │ 1902-05-08 06:32:46+08   │ 2037-04-09 13:22:35+08   │ 49280.28459490741 │
└─────────────┴────────────┴──────────────────────────┴──────────────────────────┴───────────────────┘



**期望**:2017-11-25 ~ 2017-12-03,跨度约 9 天。
如果时间戳明显超出这个范围,说明数据脏 / 时间戳单位错。

## 9. 每日记录数

In [10]:
con.sql("""
SELECT
    strftime(to_timestamp(timestamp), '%Y-%m-%d') AS date,
    COUNT(*)                                       AS rows
FROM user_behavior
GROUP BY date
ORDER BY date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────┐
│    date    │ rows  │
│  varchar   │ int64 │
├────────────┼───────┤
│ 1902-05-08 │     2 │
│ 1905-07-13 │    28 │
│ 1910-07-22 │     1 │
│ 1911-03-24 │     1 │
│ 1918-04-15 │     4 │
│ 1918-08-01 │     1 │
│ 1918-08-03 │     2 │
│ 1918-08-04 │     4 │
│ 1919-09-19 │     1 │
│ 1920-10-10 │    18 │
│     ·      │     · │
│     ·      │     · │
│     ·      │     · │
│ 2026-10-15 │     3 │
│ 2026-10-21 │     2 │
│ 2026-10-25 │     6 │
│ 2028-10-24 │     8 │
│ 2030-01-16 │     4 │
│ 2033-03-19 │    12 │
│ 2036-10-21 │     3 │
│ 2036-10-22 │    19 │
│ 2036-10-23 │     7 │
│ 2037-04-09 │     1 │
└────────────┴───────┘
       347 rows     
      (20 shown)     



**期望**:9 天数据,每天行数大致同一量级,不应有"空白日"。
双 12 预热期(12 月 2-3 日)可能出现明显增长。

In [ ]:
# ========== 量化时间戳异常 ==========

# 期望窗口: 2017-11-25 00:00 ~ 2017-12-04 00:00 (+08, 左闭右开)
# 对应 Unix 秒级时间戳:
WINDOW_START = 1511539200  # 2017-11-25 00:00:00 +08
WINDOW_END   = 1512316800  # 2017-12-04 00:00:00 +08

print('--- A. 窗口内 vs 窗口外 ---')
con.sql(f"""
SELECT
    CASE
        WHEN timestamp >= {WINDOW_START} AND timestamp < {WINDOW_END}
            THEN '窗口内(2017-11-25 ~ 12-03)'
        ELSE '窗口外(异常)'
    END                                                AS bucket,
    COUNT(*)                                           AS rows,
    COUNT(DISTINCT user_id)                            AS users,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 6) AS row_pct
FROM user_behavior
GROUP BY bucket
ORDER BY rows DESC
""").show()

print()
print('--- B. 9 个有效日期的真实分布 ---')
con.sql(f"""
SELECT
    strftime(to_timestamp(timestamp), '%Y-%m-%d') AS date,
    COUNT(*)                                       AS rows,
    COUNT(DISTINCT user_id)                        AS users
FROM user_behavior
WHERE timestamp >= {WINDOW_START} AND timestamp < {WINDOW_END}
GROUP BY date
ORDER BY date
""").show()